# 03_Futures_Database：直接使用 PyArrow 读取 Parquet

本 Notebook 演示 `futures_lake/silver` 中的八张表，并故意不调用 `c00_lakehouse.py`。每张表都在独立代码单元格中完整写出：

1. 用 `ds.partitioning(..., flavor="hive")` 声明目录分区字段；
2. 用 `ds.dataset(..., format="parquet")` 打开由多个 Parquet 文件组成的数据集；
3. 用 `dataset.to_table(columns=..., filter=...)` 执行列裁剪、分区过滤和 Parquet 扫描；
4. 按权威 Arrow Schema 校验 `pa.Table`；
5. 最后才把 Arrow Table 转换为 Pandas 用于查看。

这里专注于读取；生产写入、Hive 分区和写后复读逻辑直接位于 `a01_Data_Collection` 的 c01-c08 工作流及项目级契约模块。请使用项目标准 `latitude` Conda 环境运行。

In [1]:
from __future__ import annotations

import pathlib
import sys
from datetime import date

import pyarrow as pa
import pyarrow.dataset as ds
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()  # 当前工作目录

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

PROJECT_ROOT = candidate_root
LAKE_ROOT = PROJECT_ROOT / "03_Futures_Database" / "futures_lake" / "silver"

from config.data_contracts import (
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    FUTURES_DAILY_SCHEMA,
    FUTURES_FETCH_STATUS_SCHEMA,
    FUTURES_MINUTE_SCHEMA,
    FUTURES_MISSING_BAR_SCHEMA,
    FUTURES_SESSION_SCHEDULE_SIGNAL_SCHEMA,
    FUTURES_VARIETY_CALENDAR_SCHEMA,
    TRADE_CALENDAR_SCHEMA,
    arrow_to_pandas,
    validate_arrow_table,
)

# Hive 目录中的分区值来自路径，例如 year=2024，而不是 Parquet 文件内部。
YEAR_PARTITION_SCHEMA = pa.schema([pa.field("year", pa.int16())])
YEAR_PARTITIONING = ds.partitioning(YEAR_PARTITION_SCHEMA, flavor="hive")

EXCHANGE_MONTH_PARTITION_SCHEMA = pa.schema(
    [
        pa.field("exchange_code", pa.string()),
        pa.field("year", pa.int16()),
        pa.field("month", pa.int8()),
    ]
)
EXCHANGE_MONTH_PARTITIONING = ds.partitioning(
    EXCHANGE_MONTH_PARTITION_SCHEMA,
    flavor="hive",
)

STATUS_PARTITION_SCHEMA = pa.schema(
    [
        pa.field("bar_frequency", pa.string()),
        pa.field("exchange_code", pa.string()),
        pa.field("year", pa.int16()),
        pa.field("month", pa.int8()),
    ]
)
STATUS_PARTITIONING = ds.partitioning(STATUS_PARTITION_SCHEMA, flavor="hive")

MISSING_PARTITION_SCHEMA = pa.schema(
    [
        pa.field("bar_frequency", pa.string()),
        pa.field("exchange_code", pa.string()),
        pa.field("underlying_code", pa.string()),
        pa.field("year", pa.int16()),
        pa.field("month", pa.int8()),
    ]
)
MISSING_PARTITIONING = ds.partitioning(MISSING_PARTITION_SCHEMA, flavor="hive")

MINUTE_PARTITION_SCHEMA = pa.schema(
    [
        pa.field("exchange_code", pa.string()),
        pa.field("underlying_code", pa.string()),
        pa.field("year", pa.int16()),
        pa.field("month", pa.int8()),
    ]
)
MINUTE_PARTITIONING = ds.partitioning(
    MINUTE_PARTITION_SCHEMA,
    flavor="hive",
)

def show_arrow_and_pandas(arrow_table: pa.Table, dataframe, row_count: int = 10) -> None:
    print(f"Arrow rows={arrow_table.num_rows:,}, columns={arrow_table.num_columns}")
    print(arrow_table.schema)
    display(dataframe.head(row_count))
    display(dataframe.dtypes.rename("Pandas dtype"))

print(f"project_root={PROJECT_ROOT}")
print(f"lake_root={LAKE_ROOT}")

project_root=E:\Latitude_Analytics_v2
lake_root=E:\Latitude_Analytics_v2\03_Futures_Database\futures_lake\silver


## 1. `dim_trade_calendar`：按年份分区读取

`ds.dataset()` 只构造数据集对象；`to_table()` 才扫描数据。`year == 2024` 可以直接裁掉其他年份目录。

In [ ]:
trade_path = LAKE_ROOT / "dim_trade_calendar"
trade_columns = ["calendar_date", "date_key", "is_trading_day", "weekday", "year"]
trade_read_schema = pa.schema(
    [TRADE_CALENDAR_SCHEMA.field(column) for column in trade_columns]
)
trade_dataset = ds.dataset(
    trade_path,
    format="parquet",
    partitioning=YEAR_PARTITIONING,
)
trade_filter = ds.field("year") == 2024
trade_table = trade_dataset.to_table(
    columns=trade_columns,
    filter=trade_filter,
)
trade_table = validate_arrow_table(trade_table, trade_read_schema)
trade_calendar_df = arrow_to_pandas(trade_table, trade_read_schema).sort_values(
    "calendar_date", ascending=False
)
show_arrow_and_pandas(trade_table, trade_calendar_df)

## 2. `dim_futures_variety_calendar`：分区过滤与字段过滤

`exchange_code/year/month` 是目录分区字段；`underlying_code` 是 Parquet 文件字段。它们可以组合在同一个 Arrow Filter Expression 中。

In [ ]:
variety_path = LAKE_ROOT / "dim_futures_variety_calendar"
variety_columns = [
    "underlying_code", "exchange_code", "trading_date",
    "active_contract_count", "year", "month",
]
variety_read_schema = pa.schema(
    [FUTURES_VARIETY_CALENDAR_SCHEMA.field(column) for column in variety_columns]
)
variety_dataset = ds.dataset(
    variety_path,
    format="parquet",
    partitioning=EXCHANGE_MONTH_PARTITIONING,
)
variety_filter = (
    (ds.field("exchange_code") == "XSGE")
    & (ds.field("underlying_code") == "CU")
    & (ds.field("year") == 2024)
    & (ds.field("month") == 1)
)
variety_table = variety_dataset.to_table(
    columns=variety_columns,
    filter=variety_filter,
)
variety_table = validate_arrow_table(variety_table, variety_read_schema)
variety_calendar_df = arrow_to_pandas(
    variety_table, variety_read_schema
).sort_values("trading_date")
show_arrow_and_pandas(variety_table, variety_calendar_df)

## 3. `dim_futures_contract_calendar`：只物化目标合约 Session

日期、交易所和品种条件都交给 PyArrow；不再先把整表载入 Pandas 后过滤。

In [ ]:
contract_path = LAKE_ROOT / "dim_futures_contract_calendar"
contract_columns = [
    "contract_code", "exchange_code", "underlying_code", "trading_date",
    "session_number", "session_start_at", "session_end_at",
    "is_night_session", "year", "month",
]
contract_read_schema = pa.schema(
    [FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column) for column in contract_columns]
)
contract_dataset = ds.dataset(
    contract_path,
    format="parquet",
    partitioning=EXCHANGE_MONTH_PARTITIONING,
)
contract_filter = (
    (ds.field("exchange_code") == "XSGE")
    & (ds.field("underlying_code") == "CU")
    & (ds.field("year") == 2024)
    & (ds.field("month") == 1)
    & (ds.field("trading_date") == date(2024, 1, 2))
)
contract_table = contract_dataset.to_table(
    columns=contract_columns,
    filter=contract_filter,
)
contract_table = validate_arrow_table(contract_table, contract_read_schema)
contract_calendar_df = arrow_to_pandas(
    contract_table, contract_read_schema
).sort_values(["contract_code", "session_number"])
show_arrow_and_pandas(contract_table, contract_calendar_df)

## 4. `dim_futures_session_schedule_signal`：Session 日历信号

推断信号单独读取；`inferred` 记录不直接豁免拉取。

In [ ]:
signal_path = LAKE_ROOT / "dim_futures_session_schedule_signal"
signal_columns = [
    "exchange_code", "trading_date", "session_text", "session_start_at",
    "session_end_at", "schedule_status", "signal_reason",
    "evidence_level", "evidence_source", "is_fetch_exempt", "year", "month",
]
signal_read_schema = pa.schema(
    [FUTURES_SESSION_SCHEDULE_SIGNAL_SCHEMA.field(column) for column in signal_columns]
)
signal_dataset = ds.dataset(
    signal_path, format="parquet", partitioning=EXCHANGE_MONTH_PARTITIONING
)
signal_filter = (ds.field("exchange_code") == "XSGE") & (ds.field("year") == 2024) & (ds.field("month") == 1)
signal_table = signal_dataset.to_table(columns=signal_columns, filter=signal_filter)
signal_table = validate_arrow_table(signal_table, signal_read_schema)
schedule_signal_df = arrow_to_pandas(signal_table, signal_read_schema).sort_values(
    ["trading_date", "session_start_at"]
)
show_arrow_and_pandas(signal_table, schedule_signal_df)

## 5. `fact_futures_fetch_status`：按频率第一分区读取

先用 `bar_frequency` 裁掉另一种频率，再按交易所、年、月过滤。

In [ ]:
status_path = LAKE_ROOT / "fact_futures_fetch_status"
status_columns = [
    "bar_frequency", "contract_code", "exchange_code", "underlying_code",
    "trading_date", "session_number", "is_fetch_required",
    "is_fetch_completed", "is_data_missing", "expected_bar_count",
    "actual_bar_count", "missing_bar_count", "selection_reason",
    "year", "month",
]
status_read_schema = pa.schema(
    [FUTURES_FETCH_STATUS_SCHEMA.field(column) for column in status_columns]
)
status_dataset = ds.dataset(
    status_path,
    format="parquet",
    partitioning=STATUS_PARTITIONING,
)
status_filter = (ds.field("bar_frequency") == "1m") & (ds.field("exchange_code") == "XSGE") & (ds.field("year") == 2024) & (ds.field("month") == 1)
status_table = status_dataset.to_table(
    columns=status_columns,
    filter=status_filter,
)
status_table = validate_arrow_table(status_table, status_read_schema)
fetch_status_df = arrow_to_pandas(status_table, status_read_schema).sort_values(
    ["trading_date", "contract_code", "session_number"]
)
show_arrow_and_pandas(status_table, fetch_status_df)

## 6. `fact_futures_missing_bar`：精确缺失日或分钟

缺失表同样先按 `bar_frequency` 分区，再按交易所、品种、年、月裁剪。

In [ ]:
missing_path = LAKE_ROOT / "fact_futures_missing_bar"
missing_columns = [
    "bar_frequency", "contract_code", "exchange_code", "underlying_code",
    "trading_date", "session_number", "expected_bar_at", "detected_at",
    "year", "month",
]
missing_read_schema = pa.schema(
    [FUTURES_MISSING_BAR_SCHEMA.field(column) for column in missing_columns]
)
missing_dataset = ds.dataset(
    missing_path, format="parquet", partitioning=MISSING_PARTITIONING
)
missing_filter = (ds.field("bar_frequency") == "1d") & (ds.field("exchange_code") == "XSGE") & (ds.field("underlying_code") == "CU") & (ds.field("year") == 2024) & (ds.field("month") == 1)
missing_table = missing_dataset.to_table(
    columns=missing_columns, filter=missing_filter
)
missing_table = validate_arrow_table(missing_table, missing_read_schema)
missing_bar_df = arrow_to_pandas(missing_table, missing_read_schema).sort_values(
    ["trading_date", "contract_code", "expected_bar_at"]
)
show_arrow_and_pandas(missing_table, missing_bar_df)

## 7. `fact_futures_daily`：行情列裁剪

`columns` 决定哪些 Parquet 列会被读取。未列出的 `money`、`source` 等字段不会进入 Arrow Table。

In [ ]:
daily_path = LAKE_ROOT / "fact_futures_daily"
daily_columns = [
    "contract_code", "exchange_code", "underlying_code", "trading_date",
    "open", "high", "low", "close", "volume", "open_interest",
    "has_market_data", "year", "month",
]
daily_read_schema = pa.schema(
    [FUTURES_DAILY_SCHEMA.field(column) for column in daily_columns]
)
daily_dataset = ds.dataset(
    daily_path,
    format="parquet",
    partitioning=EXCHANGE_MONTH_PARTITIONING,
)
daily_filter = (
    (ds.field("exchange_code") == "XSGE")
    & (ds.field("underlying_code") == "CU")
    & (ds.field("year") == 2024)
    & (ds.field("month") == 1)
    & ds.field("has_market_data")
)
daily_table = daily_dataset.to_table(
    columns=daily_columns,
    filter=daily_filter,
)
daily_table = validate_arrow_table(daily_table, daily_read_schema)
daily_df = arrow_to_pandas(daily_table, daily_read_schema).sort_values(
    ["trading_date", "contract_code"]
)
show_arrow_and_pandas(daily_table, daily_df)

## 8. `fact_futures_minute`：大表的分区裁剪

分钟表必须先用四个 Hive 分区字段缩小文件集合，再增加 `trading_date` 条件；否则一次 `to_table()` 可能载入整张分钟表。

In [ ]:
minute_path = LAKE_ROOT / "fact_futures_minute"
minute_columns = [
    "contract_code", "exchange_code", "underlying_code", "trading_date",
    "session_number", "bar_at", "open", "high", "low", "close",
    "volume", "open_interest", "year", "month",
]
minute_read_schema = pa.schema(
    [FUTURES_MINUTE_SCHEMA.field(column) for column in minute_columns]
)
minute_dataset = ds.dataset(
    minute_path,
    format="parquet",
    partitioning=MINUTE_PARTITIONING,
)
minute_filter = (
    (ds.field("exchange_code") == "XSGE")
    & (ds.field("underlying_code") == "CU")
    & (ds.field("year") == 2024)
    & (ds.field("month") == 1)
    & (ds.field("trading_date") == date(2024, 1, 2))
)
minute_table = minute_dataset.to_table(
    columns=minute_columns,
    filter=minute_filter,
)
minute_table = validate_arrow_table(minute_table, minute_read_schema)
minute_df = arrow_to_pandas(minute_table, minute_read_schema).sort_values(
    ["bar_at", "contract_code"]
)
show_arrow_and_pandas(minute_table, minute_df)